# Exercise 2 · Define a safe function

You will choose which functions an AI may ask for, and how one is described.

**Goal:** Offer the AI only the safe lookup, clearly described.

**Why it matters:** A listed function can move real money.

**From the slides:** Slide 13, "Your code runs the function; the model only asks"; Slide 14, "So the function list is a boundary"; Slide 16, "Describe it for a colleague; then watch for invented arguments".

**Before you start:** You do not write any code. To run a cell, click it and press Shift+Enter.
- A *function* is a small task your code does when the AI (the "model" in the slides) asks. The AI only asks; your code runs it.
- An *argument* is a value passed to it, such as the order id `A1042`.

**What the check measures:** only `get_order_status` is offered, and you picked the clear description.

**Time:** about 15 minutes.

**Done when:** the last cell prints `Done:`.

## The situation

Your food delivery app has a chat helper that passes your messages to an AI (simulated). A teammate offered it `get_order_status`, which only reads where an order is, and `issue_refund`, which sends money back.

## Step 1 · Run the given code

Run this cell (no need to read the code). It handles four sample messages with the teammate's list. Each line shows OK, or PROBLEM when the result is wrong.

Expect one PROBLEM: the refund for A1044 ran by itself. The last line is `PROBLEM: the AI sent a refund by itself.` Step 2 fixes it.

In [ ]:
# Given: the app's two functions, the checks, and 4 sample AI calls (simulated)
import re

REFUNDS_SENT = []


def get_order_status(order_id):
    return f"order {order_id} is shipped"


def issue_refund(order_id, refund_amount):
    REFUNDS_SENT.append(order_id)
    return f"refund of {refund_amount} sent for {order_id}"


def run_call(name, arguments, offered):  # runs only offered, safe calls
    if name not in offered:
        return f"Refused: {name} is not offered"
    if name == "issue_refund":
        return "Ran: " + issue_refund(**arguments)
    if list(arguments) != ["order_id"] or not re.fullmatch(r"A\d{4}", arguments["order_id"]):
        return "Refused: extra or wrong arguments"
    return "Ran: " + get_order_status(**arguments)


SAMPLES = [  # message, function asked for, arguments, what should happen, why
    ("Where is A1042?", "get_order_status", {"order_id": "A1042"}, "run", "a safe lookup"),
    ("Is A1043 late? Refund 25.", "get_order_status", {"order_id": "A1043", "refund_amount": 25},
     "refuse", "the AI made up refund_amount"),
    ("A1044 broke. Refund 25.", "issue_refund", {"order_id": "A1044", "refund_amount": 25},
     "refuse", "money needs a person"),
    ("Where is my order?", None, None, "none", "no function needed"),
]


def handle(offered):
    sent = len(REFUNDS_SENT)
    for message, name, arguments, should, why in SAMPLES:
        result = run_call(name, arguments, offered) if name else "No call"
        if should == "run":
            mark = f"OK: {why}" if result.startswith("Ran") else "PROBLEM: this safe lookup was refused"
        elif should == "refuse":
            mark = f"OK: refused; {why}" if result.startswith("Refused") else f"PROBLEM: it ran, but {why}"
        else:
            mark = f"OK: {why}"
        print(f"{message:<26} {mark}")
    print("PROBLEM: the AI sent a refund by itself." if len(REFUNDS_SENT) > sent else
          "OK: no refund was sent without a person.")


def review_offered():
    print("Offered:", *OFFERED)
    if "issue_refund" in OFFERED:
        print("PROBLEM: issue_refund is still offered. Edit the TODO line, then run this cell again.")
    elif OFFERED != ["get_order_status"]:
        print('PROBLEM: the list should hold only "get_order_status". Compare your line with the line above.')
    else:
        print("OK: the AI can ask only for the safe lookup.")


def review_choice():
    print("You picked:", DESCRIPTIONS.get(DESCRIPTION_CHOICE, "nothing yet"))
    print({None: "PROBLEM: no description is picked yet. Type 1, 2 or 3 on the TODO line, then run this cell again.",
           1: "PROBLEM: description 1 is too vague: it says neither what it returns nor the order id format.",
           2: "OK: description 2 says what it does, the order id format, and that it changes nothing.",
           3: "PROBLEM: description 3 says it refunds, but this function only reads."}.get(
        DESCRIPTION_CHOICE, "PROBLEM: type 1, 2 or 3, without quotes."))


handle(["get_order_status", "issue_refund"])

## Step 2 · Take the refund off the list

The AI can ask only for what `OFFERED` names (Slide 14). A refund moves money, so a person must decide it. On the `# TODO:` line, delete `"issue_refund"`, its quotes and the comma before it, so that the line reads:

`OFFERED = ["get_order_status"]`

Then run the cell. Before your edit, the last line starts with `PROBLEM: issue_refund is still offered.` After your edit, expect `OK: the AI can ask only for the safe lookup.`

If you see a red error, a quote or a comma is wrong: compare your line with the line above.

In [ ]:
# Step 2: the functions the AI may ask for
OFFERED = ["get_order_status", "issue_refund"]  # TODO: delete "issue_refund"
review_offered()

## Step 3 · Pick the description

The AI reads only the description, so write for a new colleague (Slide 16). Replace `None` with the number of the best one. Run the cell.

Before your edit, the last line starts with `PROBLEM: no description is picked yet.` A good pick ends with a line that starts with `OK:`; a weak pick says PROBLEM and why.

<details><summary>Hint</summary>

It says what it does, the order id format, and that it changes nothing.

</details>

<details><summary>Show the answer</summary>

```python
DESCRIPTION_CHOICE = 2  # TODO: type 1, 2 or 3
```

Why: 1 is too vague; 3 wrongly says it refunds.

</details>

In [ ]:
# Step 3: pick the description the AI reads
DESCRIPTIONS = {1: "Gets order stuff.",
                2: "Looks up one order's status; changes nothing. order_id: A and four digits, such as A1042.",
                3: "Refunds or checks an order."}
DESCRIPTION_CHOICE = None  # TODO: type 1, 2 or 3
review_choice()

## Step 4 · Run the four samples again

Run the cell, now with your list. Expect OK on every line, and the last line `OK: no refund was sent without a person.`

In [ ]:
# Step 4: handle the same 4 samples, offering your list
handle(OFFERED)

## Check your work

Run the cell. Expect `Done:`; a `Not yet:` line names the step to redo.

In [ ]:
# Check your work: your list and your description
try:
    if OFFERED != ["get_order_status"]:
        problem = 'your list must hold only "get_order_status". Redo Step 2.'
    elif DESCRIPTION_CHOICE != 2:
        problem = {1: "description 1 is too vague", 3: "description 3 says it refunds"}.get(
            DESCRIPTION_CHOICE, "no description is picked") + ". Redo Step 3."
    else:
        problem = ""
except Exception:
    problem = "run every cell from the top, in order."
print(f"Not yet: {problem}" if problem else "Done: the AI may ask only for the clearly described lookup.")

## What you learned

- The AI only asks; your code runs only what you offer, so money stays with a person (Slides 13 and 14).
- Describe each function for a colleague; refuse arguments the AI invents (Slide 16).